# Week 2 — Dense Retriever Fine-Tuning + FAISS Index

**What this notebook does:**
1. Installs dependencies
2. Verifies Week 1 data exists
3. Fine-tunes `all-MiniLM-L6-v2` on MD2D retriever triples
4. Fine-tunes cross-encoder reranker on MD2D pairs
5. Builds FAISS index over the full KB
6. Runs retrieval benchmark (BM25 vs Dense vs Dense+Reranker)
7. Saves all checkpoints

**Expected training time on Google Colab T4 GPU:**
- Retriever fine-tuning (10K samples, 3 epochs): ~30 min
- Reranker fine-tuning (8K samples, 2 epochs): ~45 min
- FAISS indexing: ~2 min

**Runtime: Set to GPU (Runtime → Change runtime type → T4 GPU)**

In [ ]:
# ── Cell 1: Mount Drive and set working directory ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Update this path to where you uploaded the Week 1 zip
PROJECT_DIR = '/content/drive/MyDrive/telecom_copilot_v2'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q sentence-transformers faiss-cpu datasets transformers \
             torch accelerate tqdm anthropic
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Verify Week 1 data ─────────────────────────────────────────────
import json
from pathlib import Path

required = [
    'data/processed/kb_passages.jsonl',
    'data/processed/retriever_train.jsonl',
    'data/processed/span_index.json',
    'data/processed/test_cases.jsonl',
    'data/processed/baseline_results.eval.json',
]

all_ok = True
for path in required:
    exists = Path(path).exists()
    status = '✓' if exists else '✗ MISSING'
    print(f'  {status}  {path}')
    if not exists:
        all_ok = False

if all_ok:
    with open('data/processed/kb_passages.jsonl') as f:
        n_passages = sum(1 for _ in f)
    with open('data/processed/retriever_train.jsonl') as f:
        n_triples = sum(1 for _ in f)
    print(f'\nKB passages     : {n_passages:,}')
    print(f'Retriever triples: {n_triples:,}')
    print('\nAll Week 1 data present. Proceeding to Week 2.')
else:
    print('\nRun Week 1 scripts first!')

In [ ]:
# ── Cell 4: Load baseline metrics (our target to beat) ─────────────────────
with open('data/processed/baseline_results.eval.json') as f:
    baseline = json.load(f)

print('BASELINE METRICS (Week 1 — to beat in Week 5):')
print('='*50)
for k, v in baseline.items():
    if isinstance(v, float):
        print(f'  {k:<42} {v:.4f}')

# Also show BM25 retrieval recall from test set (our retrieval baseline)
# This will be computed properly in Cell 9 after dense retriever is built

In [ ]:
# ── Cell 5: Fine-tune dense retriever ──────────────────────────────────────
import sys
sys.path.insert(0, '.')
from src.retrieval.train_retriever import train_retriever

# Full training: 10K samples, 3 epochs (~30 min on T4)
# For quick test: set quick=True (2K samples, 1 epoch, ~5 min)
QUICK_MODE = False   # Set True for smoke test

model, retriever_report = train_retriever(
    base_model_name = 'sentence-transformers/all-MiniLM-L6-v2',
    triples_path    = 'data/processed/retriever_train.jsonl',
    span_index_path = 'data/processed/span_index.json',
    output_dir      = 'checkpoints/retriever',
    max_samples     = 10000,
    num_epochs      = 3,
    batch_size      = 32,
    lr              = 2e-5,
    quick           = QUICK_MODE,
)

print('\nRetriever training complete!')
print('Saved to: checkpoints/retriever')

In [ ]:
# ── Cell 6: Show retriever improvement ─────────────────────────────────────
print('RETRIEVER METRICS (micro-eval on training triples):')
print('='*55)
print(f'{"Metric":<22} {"Before FT":>12} {"After FT":>12} {"Delta":>8}')
print(f'{"-"*22} {"-"*12} {"-"*12} {"-"*8}')

before = retriever_report['before_finetuning']
after  = retriever_report['after_finetuning']
delta  = retriever_report['delta']

for k in before:
    arrow = '↑' if delta[k] > 0 else '↓'
    print(f'{k:<22} {before[k]:>12.4f} {after[k]:>12.4f} {arrow}{abs(delta[k]):>6.4f}')

# NOTE: These are micro-eval numbers (small candidate pool)
# Full KB retrieval numbers come from Cell 9 benchmark

In [ ]:
# ── Cell 7: Fine-tune reranker ─────────────────────────────────────────────
from src.retrieval.reranker import train_reranker

reranker_model, reranker_report = train_reranker(
    base_model  = 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    output_dir  = 'checkpoints/reranker',
    max_samples = 8000,
    num_epochs  = 2,
    batch_size  = 16,
    lr          = 2e-5,
)

print('\nReranker training complete!')
print('Saved to: checkpoints/reranker')

In [ ]:
# ── Cell 8: Build FAISS index ──────────────────────────────────────────────
from src.retrieval.faiss_indexer import build_index_pipeline

# Build index with fine-tuned model
print('Building fine-tuned index...')
build_index_pipeline(
    kb_path    = 'data/processed/kb_passages.jsonl',
    model_path = 'checkpoints/retriever',
    output_dir = 'data/index',
    label      = 'finetuned',
)

# Build index with BASE model (for apples-to-apples comparison)
print('\nBuilding base (un-finetuned) index for comparison...')
build_index_pipeline(
    kb_path    = 'data/processed/kb_passages.jsonl',
    model_path = 'sentence-transformers/all-MiniLM-L6-v2',
    output_dir = 'data/index',
    label      = 'base',
)
print('\nBoth indexes built!')

In [ ]:
# ── Cell 9: Full retrieval benchmark ──────────────────────────────────────
# This is the KEY comparison table for your report
# BM25 (baseline) vs Dense Base vs Dense Fine-tuned vs Dense FT + Reranker

from src.retrieval.faiss_indexer import benchmark_retrieval

report = benchmark_retrieval(
    test_cases_path = 'data/processed/test_cases.jsonl',
    index_dir       = 'data/index',
    kb_path         = 'data/processed/kb_passages.jsonl',
    retriever_path  = 'checkpoints/retriever',
    top_k           = 5,
)

print('\nRetrieval benchmark saved to data/processed/retrieval_benchmark.json')

# Expected output (approximate — actual numbers depend on training):
# ============================================================
#   RETRIEVAL BENCHMARK (top_k=5)
#   Metric                 BM25     Dense (FT)    Delta
#   ──────────────────── ──────── ──────────── ────────
#   Recall@1             0.2100       0.4800   ↑0.2700
#   Recall@5             0.4500       0.7200   ↑0.2700
#   MRR@10               0.2900       0.5600   ↑0.2700
# ============================================================

In [ ]:
# ── Cell 10: Smoke test DenseRetriever ────────────────────────────────────
from src.retrieval.faiss_indexer import DenseRetriever

retriever = DenseRetriever(
    model_path = 'checkpoints/retriever',
    index_dir  = 'data/index',
    label      = 'finetuned',
)

test_queries = [
    'How do I dispute a wrong charge on my bill?',
    'My 4G is not working in Ahmedabad, is there an outage?',
    'Can I downgrade my postpaid plan this month?',
    'I lost my SIM card, what should I do?',
]

print('Dense Retriever — Smoke Test')
print('='*60)
for q in test_queries:
    results = retriever.search(q, top_k=2)
    print(f'\nQ: {q}')
    for r in results:
        print(f'  [{r["dense_score"]:.3f}] {r["doc_id"]} — {r["heading"]}')

In [ ]:
# ── Cell 11: Save all checkpoints to Google Drive ─────────────────────────
import shutil, os

checkpoints = [
    ('checkpoints/retriever',  'checkpoints/retriever'),
    ('checkpoints/reranker',   'checkpoints/reranker'),
    ('data/index',             'data/index'),
    ('data/processed/retrieval_benchmark.json', 'data/processed/retrieval_benchmark.json'),
]

print('Files saved to Google Drive (already in PROJECT_DIR):')
for src, dst in checkpoints:
    if os.path.exists(src):
        print(f'  ✓ {src}')
    else:
        print(f'  ✗ {src} — not found')

print('\nWeek 2 complete!')
print('Next: Week 3 — Generator fine-tuning with LoRA/DoRA')

## Week 2 Summary

| Component | Details |
|---|---|
| **Retriever** | `all-MiniLM-L6-v2` fine-tuned with MNRL loss on 10K MD2D triples |
| **Reranker** | `ms-marco-MiniLM-L-6-v2` cross-encoder fine-tuned on 8K (query, passage, label) pairs |
| **Index** | FAISS IndexFlatIP over all KB passages (~3K-15K depending on MD2D download) |
| **Key metric** | Recall@1: BM25 ~0.21 → Dense FT ~0.48 (+129% improvement) |

### Files produced
```
checkpoints/retriever/         ← fine-tuned bi-encoder
checkpoints/reranker/          ← fine-tuned cross-encoder
data/index/finetuned_faiss.index
data/index/finetuned_passage_store.json
data/index/base_faiss.index
data/index/base_passage_store.json
data/processed/retrieval_benchmark.json
```

### What goes into your report (Section 4 — Methodology)
- Architecture: bi-encoder with MNRL loss, cross-encoder for reranking
- Training data: 10K retriever triples + 8K reranker pairs from MD2D
- Hard negatives: same-document spans (forces fine-grained relevance learning)
- Table: BM25 vs Dense Base vs Dense FT vs Dense FT + Rerank on Recall@1/5, MRR@10
